# Exercise

Let's apply these concepts ourselves. We can build a simple foreground on top of ecoinvent:

In [ ]:
import bw2data as bd
import bw2io as bi

In [ ]:
bi.restore_project_directory(
    "<ecoinvent project file>",
    project_name="Temporal exercise",
    overwrite_existing=True,
    switch=True,
)

In [ ]:
bd.databases

In [ ]:
bd.databases['ecoinvent-3.10.1-biosphere']['static'] = True
bd.databases['ecoinvent-3.10.1-cutoff']['static'] = True
bd.databases.flush()

In [ ]:
ei = bd.Database("ecoinvent-3.10.1-cutoff")
len(ei)

You can print all of them and pick the one you like the best.

In [ ]:
sorted(bd.methods)[:10]

In [ ]:
[ic for ic in bd.methods if "gtp100" in str(ic).lower() and "ipcc 2021" in str(ic).lower()]

In [ ]:
import bw_temporalis as bwt
import bw2calc as bc
from linear_decrease import LinearDecreaseOverTime

## The task

- Create a new database
- Pick a car or truck driving process in ecoinvent
- Copy that car or truck process over to a new database
- Pick 1-3 inputs for the car or truck (motive energy, vehicle construction, etc.) and copy those over to the new database
- Relink the copied car or truck to the copied inputs by changing and saving the relevant edges
- Add temporal dynamics to the modified inputs. If you are feeling ambitious, you can also use `LinearDecreaseOverTime`, `FixedTD` or `FixedTimeOfYearTD`.
- Pick an global warming LCIA method
- Perform a temporal LCIA calculation and graph the timeline

### Some helpful methods

* ei.search("something")
* bd.get_node(name="foo", location="bar")  # The attributes can be anything you want
* my_node.production(), my_node.technosphere(), my_node.biosphere(), my_node.consumers()

In the `TemporalisLCA` class initiator, you can change:

* starting_datetime: datetime | str = "now",
* cutoff: float | None = 5e-4,
* biosphere_cutoff: float | None = 1e-6,
* max_calc: int | None = 2e3,

Here is a simple example where you can modify only the direct process without needing to also copy some inputs. Note that you would need to create another process, a functional unit, which used a certain number of vehicle kilometers over some time period.

In [ ]:
ei.search("transport, passenger car, small size, diesel, EURO 5 RER")

In [ ]:
vroom_vroom = bd.get_node(location="RER", name='transport, passenger car, small size, diesel, EURO 5')
for edge in vroom_vroom.edges():
    print(edge)

In [ ]:
my_edge = next(edge for edge in vroom_vroom.technosphere() if edge.input['name'] == 'maintenance, passenger car')
my_edge

In [ ]:
my_edge['temporal distribution'] = bwt.easy_timedelta_distribution(
    start=0,
    end=11,
    resolution="M",
    steps=12,
)

Be sure to save!

In [ ]:
my_edge.save()